# 📊 Employee Data Cleaning & Visualization Project

End-to-end HR analytics pipeline: **generate → clean → visualize → report**.

**Run all cells in order** (Runtime → Run all). Everything is self-contained — no file uploads needed.

| Step | What it does |
|---|---|
| 1 | Install/import libraries |
| 2 | Generate a raw, messy employee dataset |
| 3 | Clean the data (duplicates, outliers, missing values, feature engineering) |
| 4 | Dashboard 1 & 2 — Overview + Advanced Analytics |
| 5 | Dashboard 3 — Attrition Deep Dive |
| 6 | Dashboard 4 — Compensation & Pay Equity |
| 7 | Download all outputs as a zip |


## 1️⃣ Setup — Install & Import Libraries

In [ ]:
!pip install -q pandas matplotlib seaborn numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully ✅")


## 2️⃣ Generate Raw Dataset

Creates a synthetic employee HR dataset with **intentional data quality issues**: missing values, outliers, duplicates, and inconsistent text formatting — simulating a real-world messy dataset.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

departments = ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance', 'Operations']
cities = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Pune']
genders = ['Male', 'Female', 'Other']

data = {
    'employee_id': [f'EMP{str(i).zfill(4)}' for i in range(1, n+1)],
    'name': [f'Employee_{i}' for i in range(1, n+1)],
    'age': np.random.randint(22, 60, n).astype(float),
    'gender': np.random.choice(genders, n, p=[0.55, 0.42, 0.03]),
    'department': np.random.choice(departments, n),
    'city': np.random.choice(cities, n),
    'salary': np.random.normal(75000, 20000, n),
    'experience_years': np.random.randint(0, 35, n).astype(float),
    'performance_score': np.random.uniform(1, 5, n),
    'satisfaction_score': np.random.uniform(1, 10, n),
    'training_hours': np.random.randint(0, 100, n).astype(float),
    'projects_completed': np.random.randint(0, 20, n),
    'attrition': np.random.choice(['Yes', 'No'], n, p=[0.2, 0.8]),
}

df = pd.DataFrame(data)

# Introduce missing values (~8%)
for col in ['age', 'salary', 'experience_years', 'performance_score', 'satisfaction_score', 'training_hours']:
    mask = np.random.random(n) < 0.08
    df.loc[mask, col] = np.nan

# Introduce outliers
df.loc[np.random.choice(n, 10, replace=False), 'salary'] = np.random.uniform(250000, 500000, 10)
df.loc[np.random.choice(n, 5, replace=False), 'age'] = np.random.choice([5, 120, 150], 5)

# Introduce duplicates
dup_idx = np.random.choice(n, 15, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

# Inconsistent formatting
df.loc[np.random.choice(len(df), 20), 'gender'] = df.loc[np.random.choice(len(df), 20), 'gender'].str.upper() if False else df['gender']
df['department'] = df['department'].apply(lambda x: x.lower() if np.random.random() < 0.1 else x)

df.to_csv('raw_employee_data.csv', index=False)
print(f"Raw dataset created: {len(df)} rows, {df.shape[1]} columns")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")


## 3️⃣ Data Cleaning Pipeline

5-step cleaning process:
1. **Remove duplicates**
2. **Standardize text** (Title Case for department/gender/city)
3. **Handle outliers** (IQR clipping for salary, domain rules for age)
4. **Fill missing values** (department-wise median imputation)
5. **Feature engineering** (salary bands, age groups, experience bands, salary/experience ratio)

In [ ]:
import pandas as pd
import numpy as np

print("=" * 60)
print("STEP 1: LOADING RAW DATA")
print("=" * 60)
df = pd.read_csv('/home/claude/data_project/raw_employee_data.csv')
print(f"Shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 5 rows:\n{df.head()}")

cleaning_log = []

# ─── STEP 2: DUPLICATES ───────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: REMOVING DUPLICATES")
print("=" * 60)
dups = df.duplicated().sum()
df = df.drop_duplicates()
cleaning_log.append(f"Removed {dups} duplicate rows")
print(f"Removed {dups} duplicate rows → {len(df)} rows remain")

# ─── STEP 3: STANDARDIZE TEXT ─────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: STANDARDIZING TEXT FIELDS")
print("=" * 60)
df['department'] = df['department'].str.strip().str.title()
df['gender'] = df['gender'].str.strip().str.title()
df['city'] = df['city'].str.strip().str.title()
cleaning_log.append("Standardized text fields (department, gender, city) to Title Case")
print("Standardized: department, gender, city → Title Case")
print(f"Departments: {sorted(df['department'].unique())}")

# ─── STEP 4: OUTLIER DETECTION & REMOVAL ──────────────────────
print("\n" + "=" * 60)
print("STEP 4: HANDLING OUTLIERS")
print("=" * 60)

def remove_outliers_iqr(df, col, label=""):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    print(f"  {col}: {outliers} outliers clipped [{lower:.1f}, {upper:.1f}]")
    return df, outliers

# Age: valid range 18–65
age_out = ((df['age'] < 18) | (df['age'] > 65)).sum()
df.loc[(df['age'] < 18) | (df['age'] > 65), 'age'] = np.nan
cleaning_log.append(f"Age outliers (<18 or >65): set {age_out} to NaN")
print(f"Age: {age_out} invalid values (outside 18-65) → set to NaN")

df, sal_out = remove_outliers_iqr(df, 'salary', 'Salary')
cleaning_log.append(f"Salary: {sal_out} outliers clipped via IQR")

# ─── STEP 5: MISSING VALUES ────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: HANDLING MISSING VALUES")
print("=" * 60)
missing_before = df.isnull().sum()
print(f"Missing values before:\n{missing_before[missing_before > 0]}")

# Numerical: fill with median per department
num_cols = ['age', 'salary', 'experience_years', 'performance_score',
            'satisfaction_score', 'training_hours']
for col in num_cols:
    before = df[col].isnull().sum()
    df[col] = df.groupby('department')[col].transform(lambda x: x.fillna(x.median()))
    after = df[col].isnull().sum()
    if before > 0:
        print(f"  {col}: {before} filled with department-wise median")
        cleaning_log.append(f"{col}: {before} NaN filled with department median")

# Any remaining (if department has all NaN) → global median
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

missing_after = df.isnull().sum().sum()
print(f"\nMissing values after: {missing_after}")

# ─── STEP 6: FEATURE ENGINEERING ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: FEATURE ENGINEERING")
print("=" * 60)
df['salary_band'] = pd.cut(df['salary'],
    bins=[0, 50000, 75000, 100000, float('inf')],
    labels=['Low', 'Medium', 'High', 'Very High'])
df['age_group'] = pd.cut(df['age'],
    bins=[17, 30, 40, 50, 65],
    labels=['22-30', '31-40', '41-50', '51-65'])
df['experience_band'] = pd.cut(df['experience_years'],
    bins=[-1, 2, 7, 15, 35],
    labels=['Junior', 'Mid', 'Senior', 'Expert'])
df['salary_per_year_exp'] = (df['salary'] / (df['experience_years'] + 1)).round(2)

cleaning_log.append("Added: salary_band, age_group, experience_band, salary_per_year_exp")
print("Added: salary_band, age_group, experience_band, salary_per_year_exp")

# ─── STEP 7: SAVE CLEAN DATA ──────────────────────────────────
df.to_csv('clean_employee_data.csv', index=False)
print(f"\nClean dataset saved: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n" + "=" * 60)
print("CLEANING SUMMARY LOG")
print("=" * 60)
for i, log in enumerate(cleaning_log, 1):
    print(f"  {i}. {log}")

print("\nClean data sample:")
print(df[['employee_id','department','age','salary','salary_band','experience_band','performance_score']].head(8).to_string(index=False))


## 4️⃣ Dashboard 1 & 2 — Overview + Advanced Analytics

**Dashboard 1:** KPI cards, salary distribution, headcount by department, boxplots, scatter plots, gender/age/city breakdowns.

**Dashboard 2:** Correlation heatmap, experience vs salary, department metrics comparison, salary bands, attrition heatmap, training hours distribution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ─── THEME SETUP ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0F1117',
    'axes.facecolor': '#1A1D2E',
    'axes.edgecolor': '#2D3154',
    'axes.labelcolor': '#C8CAD8',
    'axes.titlecolor': '#E8EAF6',
    'xtick.color': '#9094A8',
    'ytick.color': '#9094A8',
    'text.color': '#C8CAD8',
    'grid.color': '#2D3154',
    'grid.alpha': 0.6,
    'font.family': 'DejaVu Sans',
    'font.size': 9,
})

COLORS = {
    'primary': '#7C83FD',
    'accent': '#96EFCF',
    'warm': '#FFA07A',
    'rose': '#FF6B9D',
    'gold': '#FFD93D',
    'teal': '#4FC3F7',
    'bg': '#0F1117',
    'panel': '#1A1D2E',
    'border': '#2D3154',
    'text': '#E8EAF6',
    'muted': '#9094A8',
}

DEPT_COLORS = {
    'Engineering': '#7C83FD',
    'Finance': '#96EFCF',
    'Hr': '#FFD93D',
    'Marketing': '#FF6B9D',
    'Operations': '#FFA07A',
    'Sales': '#4FC3F7',
}
PALETTE = list(DEPT_COLORS.values())

# ─── LOAD DATA ────────────────────────────────────────────────
df = pd.read_csv('clean_employee_data.csv')
print(f"Loaded clean data: {df.shape}")

# ══════════════════════════════════════════════════════════════
# DASHBOARD 1: OVERVIEW & DISTRIBUTIONS
# ══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(22, 18))
fig.patch.set_facecolor(COLORS['bg'])
gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.45, wspace=0.38,
                       left=0.06, right=0.97, top=0.93, bottom=0.05)

# ── TITLE ─────────────────────────────────────────────────────
fig.text(0.5, 0.965, 'Employee Analytics Dashboard', ha='center',
         fontsize=24, fontweight='bold', color=COLORS['text'])
fig.text(0.5, 0.948, 'HR Data Insights  •  503 Employees  •  6 Departments  •  6 Cities',
         ha='center', fontsize=11, color=COLORS['muted'])

# ── KPI CARDS ─────────────────────────────────────────────────
kpi_data = [
    ('Total Employees', f"{len(df):,}", COLORS['primary']),
    ('Avg Salary', f"₹{df['salary'].mean()/1000:.1f}K", COLORS['accent']),
    ('Avg Performance', f"{df['performance_score'].mean():.2f}/5", COLORS['gold']),
    ('Attrition Rate', f"{(df['attrition']=='Yes').mean()*100:.1f}%", COLORS['rose']),
]
for i, (label, value, color) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(COLORS['panel'])
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.62, value, ha='center', va='center', transform=ax.transAxes,
            fontsize=22, fontweight='bold', color=color)
    ax.text(0.5, 0.22, label, ha='center', va='center', transform=ax.transAxes,
            fontsize=9, color=COLORS['muted'], style='italic')

# ── SALARY DISTRIBUTION ───────────────────────────────────────
ax1 = fig.add_subplot(gs[1, :2])
ax1.set_facecolor(COLORS['panel'])
n_bins = 30
counts, bins, patches = ax1.hist(df['salary'], bins=n_bins, edgecolor='none', alpha=0.85)
for i, patch in enumerate(patches):
    ratio = i / n_bins
    r = int(124 + ratio * (79 - 124))
    g = int(131 + ratio * (239 - 131))
    b = int(253 + ratio * (207 - 253))
    patch.set_facecolor(f'#{r:02x}{g:02x}{b:02x}')
ax1.axvline(df['salary'].mean(), color=COLORS['gold'], lw=2, ls='--', label=f"Mean: ₹{df['salary'].mean()/1000:.0f}K")
ax1.axvline(df['salary'].median(), color=COLORS['rose'], lw=2, ls=':', label=f"Median: ₹{df['salary'].median()/1000:.0f}K")
ax1.set_title('Salary Distribution', fontsize=12, fontweight='bold', pad=10)
ax1.set_xlabel('Salary (₹)', fontsize=9); ax1.set_ylabel('Count', fontsize=9)
ax1.legend(fontsize=8, framealpha=0.2, facecolor=COLORS['panel'])
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(colors=COLORS['muted'])

# ── DEPT HEADCOUNT ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 2:])
ax2.set_facecolor(COLORS['panel'])
dept_counts = df['department'].value_counts()
bars = ax2.barh(dept_counts.index, dept_counts.values,
                color=[DEPT_COLORS.get(d, COLORS['primary']) for d in dept_counts.index],
                edgecolor='none', height=0.6)
for bar, val in zip(bars, dept_counts.values):
    ax2.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=9, color=COLORS['text'], fontweight='bold')
ax2.set_title('Headcount by Department', fontsize=12, fontweight='bold', pad=10)
ax2.set_xlabel('Number of Employees', fontsize=9)
ax2.grid(axis='x', alpha=0.3); ax2.tick_params(colors=COLORS['muted'])
ax2.spines['right'].set_visible(False); ax2.spines['top'].set_visible(False)

# ── SALARY BY DEPT (BOX) ──────────────────────────────────────
ax3 = fig.add_subplot(gs[2, :2])
ax3.set_facecolor(COLORS['panel'])
dept_order = df.groupby('department')['salary'].median().sort_values(ascending=False).index
box_data = [df[df['department'] == d]['salary'].values for d in dept_order]
bp = ax3.boxplot(box_data, patch_artist=True, vert=True,
                 medianprops=dict(color='white', linewidth=2),
                 whiskerprops=dict(color=COLORS['muted']),
                 capprops=dict(color=COLORS['muted']),
                 flierprops=dict(marker='o', markersize=3, alpha=0.4, color=COLORS['muted']))
for patch, dept in zip(bp['boxes'], dept_order):
    patch.set_facecolor(DEPT_COLORS.get(dept, COLORS['primary']))
    patch.set_alpha(0.8)
ax3.set_xticklabels(dept_order, fontsize=8, rotation=15)
ax3.set_title('Salary Distribution by Department', fontsize=12, fontweight='bold', pad=10)
ax3.set_ylabel('Salary (₹)', fontsize=9)
ax3.grid(axis='y', alpha=0.3); ax3.tick_params(colors=COLORS['muted'])

# ── PERFORMANCE VS SATISFACTION ───────────────────────────────
ax4 = fig.add_subplot(gs[2, 2:])
ax4.set_facecolor(COLORS['panel'])
for dept, color in DEPT_COLORS.items():
    mask = df['department'] == dept
    ax4.scatter(df[mask]['performance_score'], df[mask]['satisfaction_score'],
                c=color, alpha=0.55, s=30, label=dept, edgecolors='none')
# Trend line
z = np.polyfit(df['performance_score'], df['satisfaction_score'], 1)
p = np.poly1d(z)
xline = np.linspace(df['performance_score'].min(), df['performance_score'].max(), 100)
ax4.plot(xline, p(xline), '--', color=COLORS['gold'], linewidth=1.5, alpha=0.8)
ax4.set_title('Performance vs Satisfaction Score', fontsize=12, fontweight='bold', pad=10)
ax4.set_xlabel('Performance Score (1–5)', fontsize=9)
ax4.set_ylabel('Satisfaction Score (1–10)', fontsize=9)
ax4.legend(fontsize=7, framealpha=0.15, facecolor=COLORS['panel'], ncol=2,
           loc='upper left', markerscale=1.2)
ax4.grid(alpha=0.3); ax4.tick_params(colors=COLORS['muted'])

# ── GENDER DISTRIBUTION (PIE) ─────────────────────────────────
ax5 = fig.add_subplot(gs[3, 0])
ax5.set_facecolor(COLORS['panel'])
gender_counts = df['gender'].value_counts()
wedge_colors = [COLORS['primary'], COLORS['rose'], COLORS['gold']]
wedges, texts, autotexts = ax5.pie(gender_counts.values,
    labels=gender_counts.index, autopct='%1.1f%%',
    colors=wedge_colors[:len(gender_counts)],
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor=COLORS['bg'], linewidth=2))
for t in texts: t.set_color(COLORS['muted']); t.set_fontsize(8)
for at in autotexts: at.set_color('white'); at.set_fontsize(8); at.set_fontweight('bold')
ax5.set_title('Gender Distribution', fontsize=12, fontweight='bold', pad=10)

# ── ATTRITION BY DEPT ─────────────────────────────────────────
ax6 = fig.add_subplot(gs[3, 1])
ax6.set_facecolor(COLORS['panel'])
attr_dept = df.groupby('department')['attrition'].apply(lambda x: (x=='Yes').mean()*100).sort_values()
colors_attr = [COLORS['accent'] if v < 20 else COLORS['warm'] if v < 25 else COLORS['rose']
               for v in attr_dept.values]
bars6 = ax6.barh(attr_dept.index, attr_dept.values, color=colors_attr, height=0.6, edgecolor='none')
for bar, val in zip(bars6, attr_dept.values):
    ax6.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=8, color=COLORS['text'])
ax6.set_title('Attrition Rate by Dept', fontsize=12, fontweight='bold', pad=10)
ax6.set_xlabel('Attrition %', fontsize=9)
ax6.grid(axis='x', alpha=0.3); ax6.tick_params(colors=COLORS['muted'])
ax6.spines['right'].set_visible(False); ax6.spines['top'].set_visible(False)

# ── AGE GROUP DISTRIBUTION ────────────────────────────────────
ax7 = fig.add_subplot(gs[3, 2])
ax7.set_facecolor(COLORS['panel'])
age_counts = df['age_group'].value_counts().sort_index()
bars7 = ax7.bar(age_counts.index, age_counts.values,
                color=[COLORS['teal'], COLORS['primary'], COLORS['accent'], COLORS['gold']],
                edgecolor='none', width=0.6)
for bar, val in zip(bars7, age_counts.values):
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', fontsize=8, color=COLORS['text'], fontweight='bold')
ax7.set_title('Age Group Distribution', fontsize=12, fontweight='bold', pad=10)
ax7.set_ylabel('Count', fontsize=9)
ax7.grid(axis='y', alpha=0.3); ax7.tick_params(colors=COLORS['muted'])

# ── CITY DISTRIBUTION ─────────────────────────────────────────
ax8 = fig.add_subplot(gs[3, 3])
ax8.set_facecolor(COLORS['panel'])
city_counts = df['city'].value_counts()
city_colors = [COLORS['primary'], COLORS['accent'], COLORS['gold'],
               COLORS['rose'], COLORS['teal'], COLORS['warm']]
bars8 = ax8.bar(city_counts.index, city_counts.values,
                color=city_colors[:len(city_counts)], edgecolor='none', width=0.6)
for bar, val in zip(bars8, city_counts.values):
    ax8.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', fontsize=8, color=COLORS['text'], fontweight='bold')
ax8.set_title('Employees by City', fontsize=12, fontweight='bold', pad=10)
ax8.set_ylabel('Count', fontsize=9)
ax8.tick_params(axis='x', rotation=30, labelsize=8)
ax8.grid(axis='y', alpha=0.3); ax8.tick_params(colors=COLORS['muted'])

plt.savefig('dashboard_1_overview.png',
            dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
plt.close()
print("Dashboard 1 saved.")

# ══════════════════════════════════════════════════════════════
# DASHBOARD 2: ADVANCED INSIGHTS
# ══════════════════════════════════════════════════════════════
fig2 = plt.figure(figsize=(22, 18))
fig2.patch.set_facecolor(COLORS['bg'])
gs2 = gridspec.GridSpec(3, 3, figure=fig2, hspace=0.42, wspace=0.38,
                        left=0.06, right=0.97, top=0.93, bottom=0.05)

fig2.text(0.5, 0.965, 'Advanced HR Analytics', ha='center',
          fontsize=24, fontweight='bold', color=COLORS['text'])
fig2.text(0.5, 0.948, 'Compensation  •  Performance  •  Experience  •  Attrition Analysis',
          ha='center', fontsize=11, color=COLORS['muted'])

# ── CORRELATION HEATMAP ───────────────────────────────────────
ax = fig2.add_subplot(gs2[0, :2])
ax.set_facecolor(COLORS['panel'])
num_cols = ['age', 'salary', 'experience_years', 'performance_score',
            'satisfaction_score', 'training_hours', 'projects_completed']
corr = df[num_cols].corr()
col_labels = ['Age', 'Salary', 'Experience', 'Performance', 'Satisfaction', 'Training', 'Projects']
cmap = LinearSegmentedColormap.from_list('custom',
    [COLORS['rose'], COLORS['panel'], COLORS['primary']])
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(col_labels))); ax.set_yticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, rotation=30, ha='right', fontsize=8)
ax.set_yticklabels(col_labels, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.4 else COLORS['muted']
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold', pad=10)

# ── EXPERIENCE VS SALARY (SCATTER) ────────────────────────────
ax9 = fig2.add_subplot(gs2[0, 2])
ax9.set_facecolor(COLORS['panel'])
for dept, color in DEPT_COLORS.items():
    mask_d = df['department'] == dept
    ax9.scatter(df[mask_d]['experience_years'], df[mask_d]['salary'],
                c=color, alpha=0.5, s=20, label=dept)
z = np.polyfit(df['experience_years'], df['salary'], 1)
p = np.poly1d(z)
xr = np.linspace(0, df['experience_years'].max(), 100)
ax9.plot(xr, p(xr), '--', color=COLORS['gold'], lw=1.5)
ax9.set_title('Experience vs Salary', fontsize=12, fontweight='bold', pad=10)
ax9.set_xlabel('Years of Experience', fontsize=9)
ax9.set_ylabel('Salary (₹)', fontsize=9)
ax9.legend(fontsize=6, framealpha=0.15, facecolor=COLORS['panel'], ncol=1)
ax9.grid(alpha=0.3); ax9.tick_params(colors=COLORS['muted'])

# ── AVG METRICS BY DEPT (Grouped Bar) ─────────────────────────
ax10 = fig2.add_subplot(gs2[1, :2])
ax10.set_facecolor(COLORS['panel'])
dept_metrics = df.groupby('department').agg(
    avg_performance=('performance_score', 'mean'),
    avg_satisfaction=('satisfaction_score', 'mean'),
    avg_training=('training_hours', 'mean')
).reset_index()
x = np.arange(len(dept_metrics))
w = 0.25
b1 = ax10.bar(x - w, dept_metrics['avg_performance'], w, label='Performance (÷5)', color=COLORS['primary'], alpha=0.85)
b2 = ax10.bar(x,     dept_metrics['avg_satisfaction'] / 2, w, label='Satisfaction (÷10)', color=COLORS['accent'], alpha=0.85)
b3 = ax10.bar(x + w, dept_metrics['avg_training'] / 20, w, label='Training hrs (÷20)', color=COLORS['gold'], alpha=0.85)
ax10.set_xticks(x)
ax10.set_xticklabels(dept_metrics['department'], rotation=15, fontsize=8)
ax10.set_title('Department Metrics Comparison (Normalized)', fontsize=12, fontweight='bold', pad=10)
ax10.legend(fontsize=8, framealpha=0.2, facecolor=COLORS['panel'])
ax10.grid(axis='y', alpha=0.3); ax10.tick_params(colors=COLORS['muted'])
ax10.set_ylabel('Score (normalized)', fontsize=9)

# ── SALARY BAND PIE ───────────────────────────────────────────
ax11 = fig2.add_subplot(gs2[1, 2])
ax11.set_facecolor(COLORS['panel'])
sb_counts = df['salary_band'].value_counts()
wedge_c = [COLORS['teal'], COLORS['primary'], COLORS['warm'], COLORS['rose']]
wedges11, texts11, autos11 = ax11.pie(
    sb_counts.values, labels=sb_counts.index, autopct='%1.1f%%',
    colors=wedge_c[:len(sb_counts)], startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor=COLORS['bg'], linewidth=2))
for t in texts11: t.set_color(COLORS['muted']); t.set_fontsize(8)
for a in autos11: a.set_color('white'); a.set_fontsize(8); a.set_fontweight('bold')
ax11.set_title('Salary Band Distribution', fontsize=12, fontweight='bold', pad=10)

# ── ATTRITION HEATMAP (DEPT × AGE GROUP) ─────────────────────
ax12 = fig2.add_subplot(gs2[2, :2])
ax12.set_facecolor(COLORS['panel'])
pivot = df.pivot_table(values='attrition', index='department',
                       columns='age_group',
                       aggfunc=lambda x: (x == 'Yes').mean() * 100)
cmap2 = LinearSegmentedColormap.from_list('attr', [COLORS['accent'], COLORS['gold'], COLORS['rose']])
im2 = ax12.imshow(pivot.values, cmap=cmap2, aspect='auto', vmin=0, vmax=40)
ax12.set_xticks(range(len(pivot.columns))); ax12.set_yticks(range(len(pivot.index)))
ax12.set_xticklabels(pivot.columns, fontsize=9)
ax12.set_yticklabels(pivot.index, fontsize=9)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax12.text(j, i, f'{val:.0f}%', ha='center', va='center',
                      fontsize=9, color='white', fontweight='bold')
plt.colorbar(im2, ax=ax12, shrink=0.8, label='Attrition %')
ax12.set_title('Attrition Rate Heatmap: Department × Age Group', fontsize=12, fontweight='bold', pad=10)

# ── TRAINING HOURS DISTRIBUTION ───────────────────────────────
ax13 = fig2.add_subplot(gs2[2, 2])
ax13.set_facecolor(COLORS['panel'])
for dept, color in DEPT_COLORS.items():
    data = df[df['department'] == dept]['training_hours']
    ax13.hist(data, bins=15, alpha=0.45, color=color, label=dept, edgecolor='none')
ax13.set_title('Training Hours Distribution', fontsize=12, fontweight='bold', pad=10)
ax13.set_xlabel('Training Hours', fontsize=9)
ax13.set_ylabel('Frequency', fontsize=9)
ax13.legend(fontsize=7, framealpha=0.15, facecolor=COLORS['panel'])
ax13.grid(alpha=0.3); ax13.tick_params(colors=COLORS['muted'])

plt.savefig('dashboard_2_advanced.png',
            dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
plt.close()
print("Dashboard 2 saved.")

print("\nAll visualizations complete!")


## 5️⃣ Dashboard 3 — Attrition Deep Dive

Who leaves, and why? Salary/satisfaction comparisons (stayed vs left), performance violin plots, gender attrition breakdown, and a **risk-factor correlation chart** ranking what actually predicts turnover.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0F1117', 'axes.facecolor': '#1A1D2E',
    'axes.edgecolor': '#2D3154', 'axes.labelcolor': '#C8CAD8',
    'axes.titlecolor': '#E8EAF6', 'xtick.color': '#9094A8',
    'ytick.color': '#9094A8', 'text.color': '#C8CAD8',
    'grid.color': '#2D3154', 'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans', 'font.size': 9,
})

C = {
    'primary': '#7C83FD', 'accent': '#96EFCF', 'warm': '#FFA07A',
    'rose': '#FF6B9D', 'gold': '#FFD93D', 'teal': '#4FC3F7',
    'bg': '#0F1117', 'panel': '#1A1D2E', 'border': '#2D3154',
    'text': '#E8EAF6', 'muted': '#9094A8',
}
DEPT_COLORS = {
    'Engineering': '#7C83FD', 'Finance': '#96EFCF', 'Hr': '#FFD93D',
    'Marketing': '#FF6B9D', 'Operations': '#FFA07A', 'Sales': '#4FC3F7',
}

df = pd.read_csv('clean_employee_data.csv')
stayed = df[df['attrition'] == 'No']
left   = df[df['attrition'] == 'Yes']

fig = plt.figure(figsize=(22, 20))
fig.patch.set_facecolor(C['bg'])
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.48, wspace=0.38,
                       left=0.06, right=0.97, top=0.93, bottom=0.05)

fig.text(0.5, 0.965, 'Attrition & Retention Deep Dive', ha='center',
         fontsize=24, fontweight='bold', color=C['text'])
fig.text(0.5, 0.948, 'Who leaves? Why? What patterns predict turnover?',
         ha='center', fontsize=11, color=C['muted'])

# ── STORY BANNER ──────────────────────────────────────────────
ax_story = fig.add_subplot(gs[0, :])
ax_story.set_facecolor('#12152A')
ax_story.set_xticks([]); ax_story.set_yticks([])
for sp in ax_story.spines.values(): sp.set_edgecolor(C['rose']); sp.set_linewidth(1.5)

stats = [
    ('19.9%', 'Overall Attrition', C['rose']),
    (f"{left['salary'].mean()/1000:.0f}K", 'Avg Salary (Left)', C['warm']),
    (f"{stayed['salary'].mean()/1000:.0f}K", 'Avg Salary (Stayed)', C['accent']),
    (f"{left['satisfaction_score'].mean():.1f}", 'Avg Satisfaction (Left)', C['gold']),
    (f"{stayed['satisfaction_score'].mean():.1f}", 'Avg Satisfaction (Stayed)', C['teal']),
    (f"{left['performance_score'].mean():.2f}", 'Avg Performance (Left)', C['primary']),
]
for i, (val, label, color) in enumerate(stats):
    x = 0.08 + i * 0.155
    ax_story.text(x, 0.65, val, transform=ax_story.transAxes,
                  fontsize=18, fontweight='bold', color=color, ha='center')
    ax_story.text(x, 0.22, label, transform=ax_story.transAxes,
                  fontsize=7.5, color=C['muted'], ha='center')
ax_story.set_title('', pad=0)

# ── SALARY: STAYED vs LEFT (Overlapping Hist) ────────────────
ax1 = fig.add_subplot(gs[1, 0])
ax1.set_facecolor(C['panel'])
ax1.hist(stayed['salary'], bins=28, alpha=0.6, color=C['accent'], label='Stayed', edgecolor='none')
ax1.hist(left['salary'],   bins=28, alpha=0.6, color=C['rose'],   label='Left',   edgecolor='none')
ax1.axvline(stayed['salary'].mean(), color=C['accent'], lw=1.5, ls='--')
ax1.axvline(left['salary'].mean(),   color=C['rose'],   lw=1.5, ls='--')
ax1.set_title('Salary: Stayed vs Left', fontsize=11, fontweight='bold', pad=8)
ax1.set_xlabel('Salary (₹)', fontsize=9); ax1.set_ylabel('Count', fontsize=9)
ax1.legend(fontsize=8, framealpha=0.2, facecolor=C['panel'])
ax1.grid(axis='y', alpha=0.3)

# ── SATISFACTION: STAYED vs LEFT ──────────────────────────────
ax2 = fig.add_subplot(gs[1, 1])
ax2.set_facecolor(C['panel'])
ax2.hist(stayed['satisfaction_score'], bins=20, alpha=0.6, color=C['teal'], label='Stayed', edgecolor='none')
ax2.hist(left['satisfaction_score'],   bins=20, alpha=0.6, color=C['gold'], label='Left',   edgecolor='none')
ax2.set_title('Satisfaction: Stayed vs Left', fontsize=11, fontweight='bold', pad=8)
ax2.set_xlabel('Satisfaction Score (1–10)', fontsize=9); ax2.set_ylabel('Count', fontsize=9)
ax2.legend(fontsize=8, framealpha=0.2, facecolor=C['panel'])
ax2.grid(axis='y', alpha=0.3)

# ── ATTRITION BY EXPERIENCE BAND ─────────────────────────────
ax3 = fig.add_subplot(gs[1, 2])
ax3.set_facecolor(C['panel'])
exp_attr = df.groupby('experience_band')['attrition'].apply(
    lambda x: (x=='Yes').mean()*100).reset_index()
exp_attr.columns = ['band', 'rate']
bar_colors = [C['accent'], C['primary'], C['warm'], C['rose']]
bars = ax3.bar(exp_attr['band'], exp_attr['rate'],
               color=bar_colors[:len(exp_attr)], edgecolor='none', width=0.55)
for bar, val in zip(bars, exp_attr['rate']):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontsize=9, color=C['text'], fontweight='bold')
ax3.set_title('Attrition by Experience Band', fontsize=11, fontweight='bold', pad=8)
ax3.set_xlabel('Experience Level', fontsize=9); ax3.set_ylabel('Attrition %', fontsize=9)
ax3.grid(axis='y', alpha=0.3)

# ── VIOLIN: PERFORMANCE SCORE by ATTRITION ───────────────────
ax4 = fig.add_subplot(gs[2, 0])
ax4.set_facecolor(C['panel'])
groups = [stayed['performance_score'].values, left['performance_score'].values]
vp = ax4.violinplot(groups, positions=[1, 2], showmedians=True, showextrema=True)
for i, (body, color) in enumerate(zip(vp['bodies'], [C['accent'], C['rose']])):
    body.set_facecolor(color); body.set_alpha(0.65); body.set_edgecolor('none')
vp['cmedians'].set_color('white'); vp['cmedians'].set_linewidth(2)
vp['cbars'].set_color(C['muted']); vp['cmins'].set_color(C['muted']); vp['cmaxes'].set_color(C['muted'])
ax4.set_xticks([1, 2]); ax4.set_xticklabels(['Stayed', 'Left'], fontsize=10)
ax4.set_title('Performance Score Distribution', fontsize=11, fontweight='bold', pad=8)
ax4.set_ylabel('Performance Score (1–5)', fontsize=9)
ax4.grid(axis='y', alpha=0.3)

# ── TRAINING HOURS vs ATTRITION (BOX) ────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
ax5.set_facecolor(C['panel'])
groups2 = [stayed['training_hours'].values, left['training_hours'].values]
bp = ax5.boxplot(groups2, patch_artist=True, showfliers=True,
                 medianprops=dict(color='white', linewidth=2),
                 whiskerprops=dict(color=C['muted']),
                 capprops=dict(color=C['muted']),
                 flierprops=dict(marker='o', markersize=3, alpha=0.3, color=C['muted']))
bp['boxes'][0].set_facecolor(C['teal']); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor(C['warm']); bp['boxes'][1].set_alpha(0.7)
ax5.set_xticks([1, 2]); ax5.set_xticklabels(['Stayed', 'Left'], fontsize=10)
ax5.set_title('Training Hours vs Attrition', fontsize=11, fontweight='bold', pad=8)
ax5.set_ylabel('Training Hours', fontsize=9)
ax5.grid(axis='y', alpha=0.3)

# ── GENDER × ATTRITION STACKED BAR ───────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
ax6.set_facecolor(C['panel'])
gender_attr = df.groupby(['gender', 'attrition']).size().unstack(fill_value=0)
gender_attr_pct = gender_attr.div(gender_attr.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(gender_attr_pct))
for col, color in zip(['No', 'Yes'], [C['accent'], C['rose']]):
    if col in gender_attr_pct.columns:
        ax6.bar(gender_attr_pct.index, gender_attr_pct[col], bottom=bottom,
                color=color, label='Stayed' if col=='No' else 'Left',
                edgecolor='none', width=0.5)
        for i, (idx, val) in enumerate(zip(gender_attr_pct.index, gender_attr_pct[col])):
            if val > 5:
                ax6.text(i, bottom[i] + val/2, f'{val:.0f}%',
                         ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        bottom += gender_attr_pct[col].values
ax6.set_title('Attrition by Gender (%)', fontsize=11, fontweight='bold', pad=8)
ax6.set_ylabel('Percentage (%)', fontsize=9)
ax6.legend(fontsize=8, framealpha=0.2, facecolor=C['panel'])
ax6.grid(axis='y', alpha=0.3)

# ── SALARY BAND × ATTRITION HEATMAP ──────────────────────────
ax7 = fig.add_subplot(gs[3, 0])
ax7.set_facecolor(C['panel'])
pivot2 = df.pivot_table(values='attrition', index='salary_band',
                        columns='gender',
                        aggfunc=lambda x: (x=='Yes').mean()*100)
cmap3 = LinearSegmentedColormap.from_list('risk', [C['accent'], C['gold'], C['rose']])
im = ax7.imshow(pivot2.values, cmap=cmap3, aspect='auto', vmin=0, vmax=40)
ax7.set_xticks(range(len(pivot2.columns))); ax7.set_yticks(range(len(pivot2.index)))
ax7.set_xticklabels(pivot2.columns, fontsize=9)
ax7.set_yticklabels(pivot2.index, fontsize=9)
for i in range(pivot2.shape[0]):
    for j in range(pivot2.shape[1]):
        v = pivot2.values[i, j]
        if not np.isnan(v):
            ax7.text(j, i, f'{v:.0f}%', ha='center', va='center',
                     fontsize=10, color='white', fontweight='bold')
plt.colorbar(im, ax=ax7, shrink=0.85)
ax7.set_title('Attrition %: Salary Band × Gender', fontsize=11, fontweight='bold', pad=8)

# ── PROJECTS COMPLETED vs ATTRITION ──────────────────────────
ax8 = fig.add_subplot(gs[3, 1])
ax8.set_facecolor(C['panel'])
proj_attr = df.groupby('projects_completed')['attrition'].apply(
    lambda x: (x=='Yes').mean()*100)
ax8.plot(proj_attr.index, proj_attr.values, color=C['primary'], lw=2.5, marker='o',
         markersize=5, markerfacecolor=C['rose'], markeredgecolor='none')
ax8.fill_between(proj_attr.index, proj_attr.values, alpha=0.15, color=C['primary'])
ax8.set_title('Projects Completed vs Attrition Rate', fontsize=11, fontweight='bold', pad=8)
ax8.set_xlabel('Projects Completed', fontsize=9)
ax8.set_ylabel('Attrition Rate (%)', fontsize=9)
ax8.grid(alpha=0.3)

# ── TOP RISK FACTORS (Horizontal Ranked Bar) ──────────────────
ax9 = fig.add_subplot(gs[3, 2])
ax9.set_facecolor(C['panel'])
# Compute risk: correlation of numerical cols with attrition (binary)
df_tmp = df.copy()
df_tmp['attrition_bin'] = (df_tmp['attrition'] == 'Yes').astype(int)
num_features = ['salary', 'satisfaction_score', 'performance_score',
                'training_hours', 'experience_years', 'age', 'projects_completed']
corrs = df_tmp[num_features + ['attrition_bin']].corr()['attrition_bin'][num_features]
corrs_sorted = corrs.abs().sort_values()
labels = ['Salary', 'Satisfaction', 'Performance', 'Training Hrs',
          'Experience', 'Age', 'Projects']
label_map = dict(zip(num_features, labels))
bar_vals  = [corrs[f] for f in corrs_sorted.index]
bar_lbls  = [label_map[f] for f in corrs_sorted.index]
bar_cols  = [C['rose'] if v > 0 else C['accent'] for v in bar_vals]
bars9 = ax9.barh(bar_lbls, [abs(v) for v in bar_vals], color=bar_cols,
                 edgecolor='none', height=0.55)
for bar, raw in zip(bars9, bar_vals):
    sign = '+' if raw > 0 else '–'
    ax9.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
             f'{sign}{abs(raw):.3f}', va='center', fontsize=8, color=C['text'])
ax9.set_title('Attrition Risk Factors (Correlation)', fontsize=11, fontweight='bold', pad=8)
ax9.set_xlabel('|Correlation| with Attrition', fontsize=9)
pos_patch = mpatches.Patch(color=C['rose'], label='Increases Risk')
neg_patch = mpatches.Patch(color=C['accent'], label='Decreases Risk')
ax9.legend(handles=[pos_patch, neg_patch], fontsize=7.5, framealpha=0.2, facecolor=C['panel'])
ax9.grid(axis='x', alpha=0.3)
ax9.spines['right'].set_visible(False); ax9.spines['top'].set_visible(False)

plt.savefig('dashboard_3_attrition.png',
            dpi=150, bbox_inches='tight', facecolor=C['bg'])
plt.show()
plt.close()
print("Dashboard 3 saved.")


## 6️⃣ Dashboard 4 — Compensation & Pay Equity

Salary heatmaps by department/city, performance rankings, training ROI on performance, and **gender pay gap analysis** by department.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0F1117', 'axes.facecolor': '#1A1D2E',
    'axes.edgecolor': '#2D3154', 'axes.labelcolor': '#C8CAD8',
    'axes.titlecolor': '#E8EAF6', 'xtick.color': '#9094A8',
    'ytick.color': '#9094A8', 'text.color': '#C8CAD8',
    'grid.color': '#2D3154', 'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans', 'font.size': 9,
})
C = {
    'primary': '#7C83FD', 'accent': '#96EFCF', 'warm': '#FFA07A',
    'rose': '#FF6B9D', 'gold': '#FFD93D', 'teal': '#4FC3F7',
    'bg': '#0F1117', 'panel': '#1A1D2E', 'text': '#E8EAF6', 'muted': '#9094A8',
}
DEPT_COLORS = {
    'Engineering': '#7C83FD', 'Finance': '#96EFCF', 'Hr': '#FFD93D',
    'Marketing': '#FF6B9D', 'Operations': '#FFA07A', 'Sales': '#4FC3F7',
}

df = pd.read_csv('clean_employee_data.csv')

fig = plt.figure(figsize=(22, 20))
fig.patch.set_facecolor(C['bg'])
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.48, wspace=0.4,
                       left=0.06, right=0.97, top=0.93, bottom=0.05)

fig.text(0.5, 0.965, 'Compensation & Performance Analytics', ha='center',
         fontsize=24, fontweight='bold', color=C['text'])
fig.text(0.5, 0.948, 'Pay equity, performance benchmarks, and productivity insights',
         ha='center', fontsize=11, color=C['muted'])

# ── SALARY HEATMAP: DEPT × CITY ───────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor(C['panel'])
pivot = df.pivot_table(values='salary', index='department', columns='city', aggfunc='mean')
cmap = LinearSegmentedColormap.from_list('pay', ['#1A1D2E', '#4FC3F7', '#96EFCF'])
im = ax1.imshow(pivot.values, cmap=cmap, aspect='auto')
ax1.set_xticks(range(len(pivot.columns))); ax1.set_yticks(range(len(pivot.index)))
ax1.set_xticklabels(pivot.columns, fontsize=9)
ax1.set_yticklabels(pivot.index, fontsize=9)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax1.text(j, i, f'₹{v/1000:.0f}K', ha='center', va='center',
                     fontsize=9, color='white', fontweight='bold')
plt.colorbar(im, ax=ax1, shrink=0.85, label='Avg Salary (₹)')
ax1.set_title('Average Salary Heatmap: Department × City', fontsize=12, fontweight='bold', pad=10)

# ── SALARY PERCENTILE BY DEPT ─────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor(C['panel'])
depts = sorted(df['department'].unique())
p25 = [df[df['department']==d]['salary'].quantile(0.25) for d in depts]
p50 = [df[df['department']==d]['salary'].quantile(0.50) for d in depts]
p75 = [df[df['department']==d]['salary'].quantile(0.75) for d in depts]
y = np.arange(len(depts))
ax2.barh(y, p75, color=C['primary'], alpha=0.3, label='75th %ile', height=0.5)
ax2.barh(y, p50, color=C['primary'], alpha=0.7, label='Median', height=0.5)
ax2.barh(y, p25, color=C['panel'], height=0.5)
for i, (lo, mid, hi) in enumerate(zip(p25, p50, p75)):
    ax2.plot([lo, hi], [i, i], color=C['muted'], lw=1, zorder=0)
    ax2.scatter([lo, hi], [i, i], color=C['accent'], s=20, zorder=2)
    ax2.text(hi + 500, i, f'₹{hi/1000:.0f}K', va='center', fontsize=7.5, color=C['muted'])
ax2.set_yticks(y); ax2.set_yticklabels(depts, fontsize=8)
ax2.set_title('Salary Percentile by Department', fontsize=11, fontweight='bold', pad=8)
ax2.set_xlabel('Salary (₹)', fontsize=9)
ax2.legend(fontsize=7.5, framealpha=0.2, facecolor=C['panel'])
ax2.grid(axis='x', alpha=0.3)

# ── PERFORMANCE SCORE RANKING BY DEPT ────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor(C['panel'])
perf_dept = df.groupby('department')['performance_score'].mean().sort_values()
colors_p = [DEPT_COLORS.get(d, C['primary']) for d in perf_dept.index]
bars3 = ax3.barh(perf_dept.index, perf_dept.values, color=colors_p, height=0.55, edgecolor='none')
for bar, val in zip(bars3, perf_dept.values):
    ax3.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=9, color=C['text'], fontweight='bold')
ax3.axvline(df['performance_score'].mean(), color=C['gold'], lw=1.5, ls='--',
            label=f"Avg: {df['performance_score'].mean():.2f}")
ax3.set_title('Avg Performance Score by Dept', fontsize=11, fontweight='bold', pad=8)
ax3.set_xlabel('Performance Score (1–5)', fontsize=9)
ax3.legend(fontsize=8, framealpha=0.2, facecolor=C['panel'])
ax3.grid(axis='x', alpha=0.3)

# ── PROJECTS × PERFORMANCE BUBBLE ────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor(C['panel'])
for dept, color in DEPT_COLORS.items():
    mask = df['department'] == dept
    ax4.scatter(df[mask]['projects_completed'], df[mask]['performance_score'],
                s=df[mask]['training_hours'] * 0.8 + 10,
                c=color, alpha=0.45, edgecolors='none', label=dept)
ax4.set_title('Projects vs Performance\n(bubble = training hours)', fontsize=11, fontweight='bold', pad=8)
ax4.set_xlabel('Projects Completed', fontsize=9)
ax4.set_ylabel('Performance Score', fontsize=9)
ax4.legend(fontsize=6.5, framealpha=0.15, facecolor=C['panel'], ncol=2)
ax4.grid(alpha=0.3)

# ── SALARY/EXP RATIO BY DEPT ──────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor(C['panel'])
ratio = df.groupby('department')['salary_per_year_exp'].mean().sort_values(ascending=False)
colors5 = [DEPT_COLORS.get(d, C['primary']) for d in ratio.index]
bars5 = ax5.bar(ratio.index, ratio.values, color=colors5, edgecolor='none', width=0.55)
for bar, val in zip(bars5, ratio.values):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'₹{val:,.0f}', ha='center', fontsize=7.5, color=C['text'], fontweight='bold')
ax5.set_title('Avg Salary per Year of Experience', fontsize=11, fontweight='bold', pad=8)
ax5.set_ylabel('₹ / Year of Exp', fontsize=9)
ax5.tick_params(axis='x', rotation=20, labelsize=8)
ax5.grid(axis='y', alpha=0.3)

# ── TOP/BOTTOM PERFORMERS SALARY COMPARISON ───────────────────
ax6 = fig.add_subplot(gs[2, :2])
ax6.set_facecolor(C['panel'])
df['perf_tier'] = pd.cut(df['performance_score'],
    bins=[0, 2, 3.5, 5], labels=['Low (1-2)', 'Mid (2-3.5)', 'High (3.5-5)'])
tier_salary = df.groupby(['department', 'perf_tier'])['salary'].mean().unstack()
x = np.arange(len(tier_salary.index))
w = 0.25
tier_colors = [C['rose'], C['gold'], C['accent']]
for i, (tier, color) in enumerate(zip(tier_salary.columns, tier_colors)):
    if tier in tier_salary.columns:
        offset = (i - 1) * w
        bars6 = ax6.bar(x + offset, tier_salary[tier], w,
                        label=str(tier), color=color, alpha=0.85, edgecolor='none')
ax6.set_xticks(x); ax6.set_xticklabels(tier_salary.index, fontsize=8, rotation=15)
ax6.set_title('Salary by Performance Tier per Department', fontsize=12, fontweight='bold', pad=10)
ax6.set_ylabel('Average Salary (₹)', fontsize=9)
ax6.legend(title='Perf Tier', fontsize=8, framealpha=0.2, facecolor=C['panel'])
ax6.grid(axis='y', alpha=0.3)

# ── SATISFACTION DISTRIBUTION BY SALARY BAND ─────────────────
ax7 = fig.add_subplot(gs[2, 2])
ax7.set_facecolor(C['panel'])
bands = ['Low', 'Medium', 'High', 'Very High']
band_colors = [C['teal'], C['primary'], C['warm'], C['rose']]
for band, color in zip(bands, band_colors):
    data = df[df['salary_band'] == band]['satisfaction_score']
    if len(data) > 0:
        ax7.hist(data, bins=15, alpha=0.55, color=color, label=band, edgecolor='none')
ax7.set_title('Satisfaction by Salary Band', fontsize=11, fontweight='bold', pad=8)
ax7.set_xlabel('Satisfaction Score', fontsize=9)
ax7.set_ylabel('Count', fontsize=9)
ax7.legend(title='Salary Band', fontsize=7.5, framealpha=0.2, facecolor=C['panel'])
ax7.grid(axis='y', alpha=0.3)

# ── TRAINING EFFECTIVENESS: Training vs Performance ────────────
ax8 = fig.add_subplot(gs[3, 0])
ax8.set_facecolor(C['panel'])
bins_t = [0, 20, 40, 60, 80, 100]
df['training_bin'] = pd.cut(df['training_hours'], bins=bins_t,
    labels=['0-20', '21-40', '41-60', '61-80', '81-100'])
train_perf = df.groupby('training_bin')['performance_score'].mean()
ax8.plot(range(len(train_perf)), train_perf.values, color=C['gold'], lw=2.5,
         marker='D', markersize=8, markerfacecolor=C['rose'], markeredgecolor='none')
ax8.fill_between(range(len(train_perf)), train_perf.values,
                 alpha=0.15, color=C['gold'])
ax8.set_xticks(range(len(train_perf)))
ax8.set_xticklabels(train_perf.index, fontsize=8, rotation=10)
ax8.set_title('Training Hours → Performance', fontsize=11, fontweight='bold', pad=8)
ax8.set_xlabel('Training Hours Bucket', fontsize=9)
ax8.set_ylabel('Avg Performance Score', fontsize=9)
ax8.grid(alpha=0.3)

# ── GENDER PAY GAP BY DEPT ────────────────────────────────────
ax9 = fig.add_subplot(gs[3, 1:])
ax9.set_facecolor(C['panel'])
pay_gap = df.groupby(['department', 'gender'])['salary'].mean().unstack()
if 'Male' in pay_gap and 'Female' in pay_gap:
    pay_gap['gap_pct'] = ((pay_gap['Male'] - pay_gap['Female']) / pay_gap['Female'] * 100).round(1)
    gap_sorted = pay_gap['gap_pct'].sort_values()
    colors9 = [C['rose'] if v > 0 else C['accent'] for v in gap_sorted.values]
    bars9 = ax9.barh(gap_sorted.index, gap_sorted.values, color=colors9, height=0.55, edgecolor='none')
    ax9.axvline(0, color=C['muted'], lw=1.2, ls='-')
    for bar, val in zip(bars9, gap_sorted.values):
        xpos = val + 0.2 if val >= 0 else val - 0.2
        align = 'left' if val >= 0 else 'right'
        ax9.text(xpos, bar.get_y() + bar.get_height()/2,
                 f'{val:+.1f}%', va='center', ha=align, fontsize=9,
                 color=C['text'], fontweight='bold')
    ax9.set_title('Gender Pay Gap by Department (Male vs Female %)', fontsize=12, fontweight='bold', pad=10)
    ax9.set_xlabel('Pay Gap % (positive = male earns more)', fontsize=9)
    ax9.grid(axis='x', alpha=0.3)
    import matplotlib.patches as mpatches
    pos_p = mpatches.Patch(color=C['rose'], label='Male earns more')
    neg_p = mpatches.Patch(color=C['accent'], label='Female earns more')
    ax9.legend(handles=[pos_p, neg_p], fontsize=8, framealpha=0.2, facecolor=C['panel'])

plt.savefig('dashboard_4_compensation.png',
            dpi=150, bbox_inches='tight', facecolor=C['bg'])
plt.show()
plt.close()
print("Dashboard 4 saved.")


## 7️⃣ Final Summary Report

In [ ]:
df_raw = pd.read_csv('raw_employee_data.csv')
df_clean = pd.read_csv('clean_employee_data.csv')

print(f'''
┌─────────────────────────────────────────────────────┐
│            DATA CLEANING SUMMARY                    │
├─────────────────────────────────────────────────────┤
│  Raw Dataset        : {len(df_raw):>5} rows × {df_raw.shape[1]:>2} columns       │
│  Clean Dataset      : {len(df_clean):>5} rows × {df_clean.shape[1]:>2} columns       │
│  Duplicates Removed : {len(df_raw)-len(df_clean):>5} rows                      │
│  Missing Values     : ALL resolved (dept-wise median) │
│  Outliers           : Clipped via IQR method          │
│  New Features Added : 4 engineered columns            │
└─────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────┐
│            KEY INSIGHTS DISCOVERED                  │
├─────────────────────────────────────────────────────┤
│  Avg Salary         : ₹{df_clean['salary'].mean():>8,.0f}                │
│  Top Paid Dept      : {df_clean.groupby('department')['salary'].mean().idxmax():<20}         │
│  Best Performing    : {df_clean.groupby('department')['performance_score'].mean().idxmax():<20}         │
│  Highest Attrition  : {df_clean.groupby('department')['attrition'].apply(lambda x: (x=='Yes').mean()).idxmax():<20}         │
│  Attrition Rate     : {(df_clean['attrition']=='Yes').mean()*100:.1f}%                        │
│  Avg Experience     : {df_clean['experience_years'].mean():.1f} years                  │
└─────────────────────────────────────────────────────┘
''')


## 8️⃣ Download All Outputs

Zips and downloads the cleaned dataset + all 4 dashboard images.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('employee_analytics_outputs', 'zip', '.', 
    base_dir=None)

# Only zip relevant output files
import zipfile
output_files = [
    'raw_employee_data.csv',
    'clean_employee_data.csv',
    'dashboard_1_overview.png',
    'dashboard_2_advanced.png',
    'dashboard_3_attrition.png',
    'dashboard_4_compensation.png',
]
with zipfile.ZipFile('employee_analytics_outputs.zip', 'w') as zf:
    for f in output_files:
        zf.write(f)

files.download('employee_analytics_outputs.zip')
print("Download started — check your browser downloads!")
